In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

batch_df = (spark.table("dbw_agentops.bronze.agent_traces")
    .withColumn("event_time", F.col("event_time").cast("string"))
    .withColumn("ingest_time", F.col("ingest_time").cast("string")))
stream_df = (spark.table("dbw_agentops.bronze.agent_traces_stream")
    .withColumn("_source_file", F.lit("eventhubs")))

bronze = batch_df.unionByName(stream_df, allowMissingColumns=True)

# 1. Type the timestamps
typed = (bronze
    .withColumn("event_time", F.to_timestamp("event_time"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time")))

# 2. Dedupe: keep the latest-ingested copy of each trace_id
w = Window.partitionBy("trace_id").orderBy(F.col("_ingested_at").desc())
deduped = (typed.withColumn("_rn", F.row_number().over(w))
                .filter("_rn = 1").drop("_rn"))

# 3. Data contract: named rules, violations recorded per row
rules = {
    "missing_user_id": F.col("user_id").isNull(),
    "missing_model":   F.col("model").isNull(),
}
violations = F.concat_ws(",", *[F.when(cond, F.lit(name)) for name, cond in rules.items()])
checked = deduped.withColumn("_violations", violations)

good = (checked.filter(F.col("_violations") == "").drop("_violations")
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("total_tokens", F.col("prompt_tokens") + F.col("completion_tokens")))
bad = checked.filter(F.col("_violations") != "")

# 4. Create the silver table if missing, then MERGE (idempotent)
good.limit(0).write.mode("ignore").saveAsTable("dbw_agentops.silver.agent_traces")
good.createOrReplaceTempView("batch")
spark.sql("""
    MERGE INTO dbw_agentops.silver.agent_traces t
    USING batch s ON t.trace_id = s.trace_id
    WHEN NOT MATCHED THEN INSERT *
""")

# 5. Quarantine: rebuilt each run from the same inputs
bad.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbw_agentops.silver.agent_traces_quarantine")

# Reconciliation
print("silver:    ", spark.table("dbw_agentops.silver.agent_traces").count())
print("quarantine:", spark.table("dbw_agentops.silver.agent_traces_quarantine").count())